# External data inventory (Stage 1)

**Purpose:** Observation-only profile of everything under `data/` plus selected processed course tables. No joins, no flood hypotheses.

**Regenerate catalogue:** from repo root run `python -m archive.impact_eda.impact_inventory`.

Labels in this notebook:
- **Observation** — what files and the profiler report
- **Unknown** — not verified from data in this pass

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_inventory.py").exists():
    ROOT = ROOT.parents[2]  # if opened from eda/

OUT = ROOT / "eda" / "outputs" / "impact_eda"
CATALOGUE = OUT / "catalogue.csv"
DRIFT = OUT / "schema_drift_notes.txt"
SUMMARY = OUT / "inventory_summary.json"

assert CATALOGUE.exists(), "Run: python -m archive.impact_eda.impact_inventory"

cat = pd.read_csv(CATALOGUE)
with open(SUMMARY, encoding="utf-8") as f:
    summary = json.load(f)
drift_text = DRIFT.read_text(encoding="utf-8") if DRIFT.exists() else ""

summary

## Observation: families on disk

External folders under `data/` (verified by live listing during plan): seven families. Course hazard/exposure layers are referenced via `processed-data/` rows in the catalogue.

In [ ]:
ext_families = [
    "FEWS_crop_data",
    "UCDP_GED",
    "UCDP_NonState",
    "GeoEPR_2021",
    "ACLED_aggregated",
    "IOM_DTM_mobility",
    "IOM_DTM_flow",
    "OCHA_flood_data",
]
present = cat["family"].unique().tolist()
for fam in ext_families:
    n = (cat["family"] == fam).sum()
    print(f"{fam}: {n} catalogue rows")

## Observation: grain and time coverage (external only)

One row per profiled sheet/table. Files listed but not fully parsed appear with empty `n_rows_sampled`.

In [ ]:
ext = cat[cat["family"].isin(ext_families)].copy()
cols = [
    "family",
    "source_path",
    "sheet_or_table",
    "grain",
    "time_min",
    "time_max",
    "south_sudan_scope",
    "n_rows_sampled",
    "n_cols",
]
ext[cols].sort_values(["family", "source_path", "sheet_or_table"]).head(40)

In [ ]:
ext.groupby("family", dropna=False).agg(
    tables=("sheet_or_table", "count"),
    profiled_rows=("n_rows_sampled", lambda s: s.notna().sum()),
    read_errors=("read_error", lambda s: (s.astype(str).str.len() > 0).sum()),
)

## Observation: schema drift (DTM mobility / flow)

Mobility rounds use different sheet layouts; only rounds 2, 8, 12, and 16 are fully sheet-profiled. Other round files are listed with sheet names only.

In [ ]:
print(drift_text[:8000] if len(drift_text) > 8000 else drift_text)

## Observation: spatial and temporal field names

Useful for Stage 3 crosswalks — **not** proof that joins are valid.

In [ ]:
key_tables = ext[
    ext["sheet_or_table"].str.contains(
        "Baseline|Loc_Dataset|crop_data|GEDEvent|NonState|GeoEPR|Sheet1|Floods_affected|Summary",
        case=False,
        na=False,
    )
]
key_tables[["family", "sheet_or_table", "spatial_fields", "temporal_fields", "grain"]]

## Observation: read errors

Stage 1 gate: every family has a row; errors must be empty or explicitly listed.

In [ ]:
err = cat[cat["read_error"].astype(str).str.len() > 0]
if len(err) == 0:
    print("Observation: no read errors in catalogue.")
else:
    display(err[["family", "source_path", "sheet_or_table", "read_error"]])

## Observation: course processed tables (comparison only)

These support later joins to flood masks, ASAP, and IPC; they are not part of `data/`.

In [ ]:
course = cat[cat["family"].str.startswith("course_")]
course[["family", "source_path", "grain", "n_rows_sampled", "temporal_fields"]]

## Unknown / not in `data/` (cannot use until acquired)

- Event-level ACLED (only weekly admin1 aggregates are present)
- REACH / IMPACT AoK
- FEWS livelihood zones, MODIS NDVI, WFP prices, CHIRPS, AFCD raster

## Stage 1 gate status

- Catalogue: `eda/outputs/impact_eda/catalogue.csv`
- Drift notes: `eda/outputs/impact_eda/schema_drift_notes.txt`
- **Do not start Stage 2 semantics notebook until this catalogue is reviewed.**

Path helper: `processing_data/paths.py` (`COURSE_RAW`, `EXT_DATA`, `resolve_raw_data_dir()`).